# Day 037 · 多重索引与 groupby · 中国版
**MultiIndex & GroupBy** · 阶段 P2 · Python 量化工具栈

> 把一堆股票按行业分组、各算各的因子,是量化研究每天都在做的事。这一节用真实的多只跨行业 A 股,带你把 MultiIndex 多层索引、groupby 分组、stack/unstack 长宽互转、pivot_table 透视、行业中性化这五件套一次打通,顺便看穿"买了不同行业就等于分散"的错觉。

---

### 关于「中国版」

本 notebook 是为**国内学员**优化的版本:
- 数据源用 **akshare**(国内可访问、零 VPN、免注册),取代了视频里的 yfinance
- 标的尽量保持原意:美股 ETF→A 股 ETF / 国际公司→A 股龙头
- 所讲的**概念和方法 100% 一致**,但**具体数字可能与视频里略有差异**(因为是不同时间窗 / 不同标的)
- 一般情况国内 `pip install akshare` 即可,无需 token / VPN

**课件生成日期:** 2026-05-30  ·  **建议学习时长:** 22 分钟

## 🔧 第一步:环境自检 + 自动安装

**第一次拿到这份 notebook,请先运行下面这一格。** 它会:
1. 检查所有必需的 Python 包(含 `akshare`),缺什么自动 `pip install` 装上
2. 注入中文字体到 matplotlib(让图标不出乱码)
3. 跑完看到 `✓ 环境就绪` 就可以继续


In [ ]:
# === 环境自检 + 自动安装(运行此单元格即可)===
import importlib, subprocess, sys, os

REQUIRED = ["akshare", "baostock", "matplotlib", "numpy", "numpy_financial", "pandas", "scipy", "sklearn", "statsmodels"]
PIP_NAME = {"sklearn":"scikit-learn","cv2":"opencv-python","PIL":"Pillow","bs4":"beautifulsoup4","yaml":"PyYAML"}

missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(PIP_NAME.get(mod, mod))
if missing:
    print(f"⏳ 缺少 {len(missing)} 个包,自动安装:{missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("✓ 安装完成")
else:
    print(f"✓ 所有 {len(REQUIRED)} 个必需库已就绪")

# === 中文字体配置 ===
import matplotlib, matplotlib.pyplot as plt, matplotlib.font_manager as fm
CJK = ["/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
       "C:/Windows/Fonts/msyh.ttc","C:/Windows/Fonts/simhei.ttf",
       "/System/Library/Fonts/PingFang.ttc","/System/Library/Fonts/STHeiti Medium.ttc"]
for p in CJK:
    if os.path.exists(p):
        fm.fontManager.addfont(p)
        print(f"✓ 中文字体已加载:{os.path.basename(p)}")
        break
plt.rcParams["font.sans-serif"] = ["Noto Sans CJK JP","Microsoft YaHei","PingFang SC","SimHei","DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("✓ 环境就绪")


## 🔌 第二步:加载国内数据助手

下面这一格是**工具函数**(可以折叠,不需要修改)。它把 `yfinance` 风格的 ticker(如 `600519.SS`)自动路由到对应的 akshare 接口,提供 `get_close(ticker)` 和 `get_close_multi(tickers_dict)` 两个函数。

In [ ]:
# === 国内数据源助手(akshare 后端,不需要 VPN)===
# 这一格是工具函数,可以折叠,不需要修改。
# 它把 yfinance 风格的 ticker(如 "600519.SS" / "0700.HK" / "AAPL" / "BTC-USD")
# 自动路由到对应的 akshare 接口,统一返回 yfinance 风格的 Close DataFrame。

import re
from datetime import datetime, timedelta
import pandas as pd
import akshare as ak

_TICKER_MAP = {
    "^GSPC": ("us_index_sina", ".INX"),
    "^DJI":  ("us_index_sina", ".DJI"),
    "^IXIC": ("us_index_sina", ".IXIC"),
    "GC=F":  ("foreign_futures", "GC"),
    "SI=F":  ("foreign_futures", "SI"),
    "CL=F":  ("foreign_futures", "CL"),
    "BTC-USD": ("crypto", "BTC"),
    "ETH-USD": ("crypto", "ETH"),
}

def _retry(fn, *args, _retries=4, _wait=1.5, **kwargs):
    """akshare 上游(东方财富/新浪/Binance)偶有 RemoteDisconnected / Timeout,自动重试 4 次。
    2026-05-11 加:用户跑 cn notebook 拉 002594.SZ 时上游断连 → 整节卡死。
    每次重试间隔 _wait 秒(指数退避 1x → 1.5x → 2.25x)。
    """
    import time as _t
    last_err = None
    wait = _wait
    for i in range(_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            name = type(e).__name__
            if i == _retries - 1:
                print(f"  ✗ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次仍失败({name}),放弃")
                raise
            print(f"  ⚠ {getattr(fn,'__name__',str(fn))} 第 {i+1} 次失败({name}),{wait:.1f}s 后重试")
            _t.sleep(wait)
            wait *= 1.5

def _parse_period(period):
    end = datetime.today()
    m = re.match(r"^(\d+)\s*(y|mo|d|w)$", period.lower().strip())
    days = 365 * 3 if not m else int(m.group(1)) * {"y":365,"mo":30,"w":7,"d":1}[m.group(2)]
    return (end - timedelta(days=days+30)).strftime("%Y%m%d"), end.strftime("%Y%m%d")

def _classify(ticker):
    t = ticker.strip()
    if t in _TICKER_MAP: return _TICKER_MAP[t]
    if t.endswith((".SS",".SH",".SZ")):
        code = t.split(".")[0]
        if code.startswith(("51","159","58")) or code in ("510300","510500","510050","511010","513100"):
            return ("a_etf", code)
        if code in ("000300","000016","000905","000852","000001"):
            return ("a_index", code)
        return ("a_stock", code)
    if t.endswith(".HK"):
        return ("hk", t.split(".")[0].zfill(5))
    return ("us", t)

def _norm(df, dc, cc):
    out = df[[dc, cc]].copy()
    out[dc] = pd.to_datetime(out[dc])
    return out.set_index(dc).sort_index()[cc].astype(float).rename("Close")

def get_close(ticker, period="3y"):
    """返回某标的 Close 价格 series。后端 akshare,中国可访问。
    所有 ak.* 调用都过 _retry(4 次,指数退避)— 防东方财富/新浪上游瞬时断连。
    """
    start, end = _parse_period(period)
    kind, sym = _classify(ticker)
    if kind == "a_stock":
        return _norm(_retry(ak.stock_zh_a_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_etf":
        return _norm(_retry(ak.fund_etf_hist_em, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "a_index":
        idx_map = {"000300":"sh000300","000016":"sh000016","000905":"sh000905","000852":"sh000852","000001":"sh000001"}
        s = _norm(_retry(ak.stock_zh_index_daily_em, symbol=idx_map.get(sym, f"sh{sym}")), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "hk":
        return _norm(_retry(ak.stock_hk_hist, symbol=sym, period="daily", start_date=start, end_date=end, adjust="qfq"), "日期", "收盘")
    if kind == "us":
        s = _norm(_retry(ak.stock_us_daily, symbol=sym, adjust="qfq"), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "us_index_sina":
        s = _norm(_retry(ak.index_us_stock_sina, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "foreign_futures":
        s = _norm(_retry(ak.futures_foreign_hist, symbol=sym), "date", "close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    if kind == "crypto":
        import requests as _rq
        def _binance():
            r = _rq.get("https://api.binance.com/api/v3/klines",
                        params={"symbol": f"{sym}USDT", "interval": "1d", "limit": 1000}, timeout=15)
            r.raise_for_status()
            return r.json()
        klines = _retry(_binance)
        df = pd.DataFrame(klines, columns=["open_time","open","high","low","close","volume",
                                            "close_time","qav","trades","tbb","tbq","ignore"])
        df["date"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close"] = df["close"].astype(float)
        s = df.set_index("date").sort_index()["close"].rename("Close")
        return s.loc[pd.to_datetime(start, format="%Y%m%d"):]
    raise ValueError(f"unsupported ticker: {ticker}")

def get_close_multi(tickers, period="3y"):
    """批量取 Close,返回 DataFrame,列名是 tickers dict 的 key(中文名),按交集日期对齐。"""
    series = {name: get_close(t, period=period) for name, t in tickers.items()}
    return pd.concat(series, axis=1).sort_index()

print("✓ cn_data 助手已加载 — 用 get_close(ticker) / get_close_multi(tickers_dict) 拉数据")


## 学习目标

- 理解 MultiIndex 多层索引为什么天生适合"多只股票 × 多个日期"的面板数据
- 掌握 groupby 的 split-apply-combine 3 步心法,会按行业分组算平均收益、波动等因子
- 会用 stack 把宽表叠成长表、unstack 把长表摊成宽表,在两种格式间自如切换
- 会用 pivot_table 一句话做出"行业 × 指标"的透视汇总
- 理解行业中性化:剔除行业整体涨跌后,衡量个股自己的真本事

## 历史背景:老周的"十只好票",其实在押三个赛道

还记得第33 课买十只票照样一起亏的老王吗。他的朋友老周看完很受教,这回特意精选了十只"基本面好"的票,自觉很分散。可一段行情走下来,赚的没赚到、亏的全亏上。他把十只票按行业分组一算才发现:十只里有五只挤在白酒和新能源两个赛道,正好占一半,所谓分散其实是把鸡蛋换了几个篮子接着押。更扎心的是,他重仓三只的新能源整个赛道两年是负的,而他只配了一只的公用事业(长江电力)反而最强,两年涨了五成。这一节我们就用 groupby,把这种"伪分散"用一张表算到明明白白,再用行业中性化告诉他:哪几只票是真有本事,哪几只只是躺在好赛道上被抬起来的。



## 核心概念

下面每一节是听完视频后回头细读的内容。

### 1. MultiIndex 多层索引

就好比 Excel 里"大类 + 小类"两级表头。一只股票一条价格序列是一维的,而很多只股票 × 很多天,天然是二维面板,用(日期, 股票)两层标签一标,取数、分组都顺手。


### 2. groupby 与 split-apply-combine

简单讲就是分3 步:先按某个标签把数据拆成几堆(split,比如按行业),每堆各算各的(apply,比如算平均收益),再把结果拼回一张表(combine)。班主任按小组统计平均分,就是这个套路。


### 3. stack 与 unstack

stack 把"每列一只股票"的宽表叠成一长条(长格式),unstack 反过来把长条摊开成宽表。长格式适合分组和存储,宽格式适合画图和看走势,两者随时互转。


### 4. pivot_table 透视表

等于说就是 Excel 的数据透视表搬进代码:指定行是什么、列是什么、格子里放什么统计量(平均、求和、计数),一句话生成汇总表。


### 5. 行业中性化

同一个赛道的票会一起涨跌,这部分涨跌不算个股的本事。用 groupby 把每只票减去所在行业的平均,剩下的才是它相对同行的超额,这一步叫行业中性化,是因子选股的基本功。


## 实操:groupby 五件套 — MultiIndex 面板 / split-apply-combine / stack-unstack / pivot_table / 行业中性化

下面这段代码用 akshare 抓数据,国内零 VPN 跑通。**直接 Run All** 看结果。

**依赖:** `pip install pandas numpy matplotlib akshare statsmodels scipy`

In [ ]:
# day_037_groupby.py — 多重索引与 groupby:MultiIndex / split-apply-combine / stack-unstack / pivot_table / 行业中性化
# 用真实 A 股多只跨行业股票,按行业分组算因子,看穿"行业不同就分散"的错觉
# 数据:baostock 日线(免费、稳定、国内零翻墙)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import baostock as bs

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)

# 老周自以为分散的"十只好票"(其实挤在少数几个行业)+ 几只其他行业做对照
STOCKS = {
    'sh.600519': '茅台',   'sh.000858': '五粮液',     # 白酒
    'sz.300750': '宁德时代', 'sz.002594': '比亚迪',     # 新能源
    'sh.600036': '招商银行', 'sh.601318': '中国平安',   # 金融
    'sh.600276': '恒瑞医药', 'sz.000661': '长春高新',   # 医药
    'sh.600900': '长江电力',                           # 公用事业
    'sh.601012': '隆基绿能',                           # 光伏(归新能源)
}
START, END = '2023-01-01', '2024-12-31'

# 手工行业标签(教学用,真实可用 baostock query_stock_industry)
INDUSTRY = {
    '茅台': '白酒', '五粮液': '白酒',
    '宁德时代': '新能源', '比亚迪': '新能源', '隆基绿能': '新能源',
    '招商银行': '金融', '中国平安': '金融',
    '恒瑞医药': '医药', '长春高新': '医药',
    '长江电力': '公用事业',
}

print('==== 0. 数据拉取(baostock 多只跨行业日线)====')
lg = bs.login()
if lg.error_code != '0':
    raise RuntimeError(f'baostock 登录失败:{lg.error_msg}')
frames = {}
for code, name in STOCKS.items():
    rs = bs.query_history_k_data_plus(
        code, 'date,close', start_date=START, end_date=END,
        frequency='d', adjustflag='2')
    rows = []
    while (rs.error_code == '0') and rs.next():
        rows.append(rs.get_row_data())
    s = pd.DataFrame(rows, columns=['date', 'close'])
    s['date'] = pd.to_datetime(s['date'])
    s['close'] = s['close'].astype(float)
    frames[name] = s.set_index('date')['close']
bs.logout()

# 铁律 39:dict 显式映射,绝不 df.columns= 赋值
prices = pd.DataFrame({name: frames[name] for name in STOCKS.values()}).dropna()
print(f'对齐后 {prices.index[0].date()} → {prices.index[-1].date()},{len(prices)} 个交易日,{prices.shape[1]} 只股票')
rets = prices.pct_change().dropna()      # 日收益宽表(行=日期,列=股票)

# ============ 1. MultiIndex:把宽表叠成面板(长格式)============
print('\n==== 1. MultiIndex 面板数据 ====')
# stack:把"每列一只股票"的宽表,叠成 (日期, 股票) 双层索引的一列长表
long = rets.stack()
long.index.names = ['日期', '股票']
print(f'宽表形状 {rets.shape} (行=日期 列=股票)  →  stack 长表 {long.shape} (双层索引一列)')
print('长表前 5 行:')
print(long.head())

# ============ 2. groupby:split-apply-combine 按行业分组 ============
print('\n==== 2. groupby 按行业分组算因子 ====')
ind_map = pd.Series(INDUSTRY, name='行业')
# 给长表贴行业标签
df_long = long.rename('日收益').reset_index()
df_long['行业'] = df_long['股票'].map(INDUSTRY)
# 每只股票两年累计收益
cum_by_stock = (1 + rets).prod() - 1
ind_of_stock = pd.Series(INDUSTRY)
ind_cum = cum_by_stock.groupby(ind_of_stock).mean().sort_values(ascending=False)
print('各行业平均累计收益(split-apply-combine):')
for ind, v in ind_cum.items():
    print(f'  {ind:6s} {v*100:6.1f}%')
ind_count = ind_of_stock.value_counts()
print('老周持仓的行业分布(只数):')
for ind, c in ind_count.items():
    print(f'  {ind:6s} {c} 只')
top2 = ind_count.head(2)
print(f'最集中的两个行业占了 {top2.sum()}/{len(STOCKS)} 只 = {top2.sum()/len(STOCKS)*100:.0f}%,所谓分散其实在押赛道')

# ============ 3. unstack:长格式 → 宽格式(行业 × 日期 透视)============
print('\n==== 3. stack / unstack 长宽互转 ====')
daily_ind = df_long.groupby(['日期', '行业'])['日收益'].mean()      # 每天每行业平均日收益(长)
ind_wide = daily_ind.unstack('行业')                               # → 行=日期 列=行业(宽)
print(f'每天每行业平均收益:长表 {daily_ind.shape} --unstack--> 宽表 {ind_wide.shape} (行=日期 列=行业)')
ind_cum_curve = (1 + ind_wide).cumprod()
print('各行业期末净值(起点 1.0):')
for ind in ind_cum_curve.columns:
    print(f'  {ind:6s} {ind_cum_curve[ind].iloc[-1]:.3f}')

# ============ 4. pivot_table:透视表 ============
print('\n==== 4. pivot_table 透视 ====')
df_long['年月'] = df_long['日期'].dt.to_period('M').astype(str)
pv = pd.pivot_table(df_long, values='日收益', index='行业', columns=None, aggfunc=['mean', 'std', 'count'])
pv.columns = ['日均收益', '日波动', '样本数']
pv = pv.sort_values('日均收益', ascending=False)
print('按行业透视(日均收益 / 日波动 / 样本数):')
print((pv.assign(日均收益=lambda x: (x['日均收益']*100).round(3),
                 日波动=lambda x: (x['日波动']*100).round(2))))

# ============ 5. 行业中性化:groupby demean ============
print('\n==== 5. 行业中性化(groupby transform demean)====')
# 个股累计收益 减去 所在行业平均 = 剔除行业 beta 后的"个股真本事"
stock_ind = ind_of_stock
ind_mean_map = cum_by_stock.groupby(stock_ind).transform('mean')
neutral = cum_by_stock - ind_mean_map           # 行业中性化后的超额
neutral = neutral.sort_values(ascending=False)
print('行业中性化后(剔除行业平均,看个股真本事)排名:')
for name, v in neutral.items():
    print(f'  {name:6s} 中性化超额 {v*100:6.1f}%  (原始 {cum_by_stock[name]*100:6.1f}%, 行业均 {ind_mean_map[name]*100:6.1f}%)')

# ============ 画图 ============
plt.rcParams['axes.unicode_minus'] = False
# 图1:各行业平均累计收益柱状
fig, ax = plt.subplots(figsize=(10, 5))
ind_cum.plot(kind='barh', ax=ax, color='#d08770')
ax.set_title('各行业平均累计收益(2023-2024)')
ax.set_xlabel('累计收益')
for i, v in enumerate(ind_cum.values):
    ax.text(v, i, f' {v*100:.0f}%', va='center')
plt.tight_layout(); plt.savefig('chart_01.png', dpi=110); plt.close()

# 图2:各行业净值曲线
fig, ax = plt.subplots(figsize=(11, 5.5))
for ind in ind_cum_curve.columns:
    ax.plot(ind_cum_curve.index, ind_cum_curve[ind], label=ind, linewidth=1.6)
ax.axhline(1.0, color='gray', ls='--', lw=0.8)
ax.set_title('各行业平均净值曲线(起点 1.0)')
ax.legend(loc='best', ncol=2)
plt.tight_layout(); plt.savefig('chart_02.png', dpi=110); plt.close()

# 图3:持仓行业分布饼图
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(ind_count.values, labels=[f'{i}\n{c}只' for i, c in ind_count.items()],
       autopct='%1.0f%%', startangle=90)
ax.set_title('老周"十只好票"的行业分布 — 其实在押赛道')
plt.tight_layout(); plt.savefig('chart_03.png', dpi=110); plt.close()

# 图4:行业中性化前后对比
fig, ax = plt.subplots(figsize=(11, 5.5))
order = neutral.index
x = np.arange(len(order))
ax.bar(x-0.2, [cum_by_stock[n]*100 for n in order], width=0.4, label='原始累计收益', color='#81a1c1')
ax.bar(x+0.2, [neutral[n]*100 for n in order], width=0.4, label='行业中性化后', color='#a3be8c')
ax.axhline(0, color='gray', lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=35, ha='right')
ax.set_ylabel('%'); ax.set_title('行业中性化:剔除行业 beta 后的个股真本事')
ax.legend()
plt.tight_layout(); plt.savefig('chart_04.png', dpi=110); plt.close()

print('\n[done] 4 张图已保存,output.txt 见上方全部打印')

## 真实市场案例

| 市场 | 标的 | 实战观察 |
| --- | --- | --- |
| 行业分组 |  | 基金经理评估持仓时,先按行业 groupby 看自己在哪些赛道超配、哪些低配,避免无意中押注单一行业 |
| 行业分组 |  | 量化选股算因子时,几乎都要先做行业中性化,否则因子排名会被强势行业整体带偏 |
| 行业分组 |  | 研究员用 pivot_table 把几百只票按"行业 × 月份"透视,一眼看出哪个赛道在哪个阶段最强 |
| 行业分组 |  | ETF 和指数编制方按行业分组做权重上限约束,防止单一行业占比过高 |
| 行业分组 |  | 风控用 groupby 按行业汇总持仓暴露,监控行业集中度是否超标 |


## 常见坑

### ⚠ 01. 误以为行业不同就是分散

真正的分散看的是相关性和行业集中度。八只票分属四个行业,但五只挤在两个赛道,本质还是押注,groupby 一算就现形。

### ⚠ 02. groupby 后忘了选聚合方式

分完组必须告诉它每组怎么算(mean/sum/std)。直接打印分组对象只会得到一个看不懂的中间态,记得跟上一个聚合函数。

### ⚠ 03. stack/unstack 弄混方向

stack 让表"变高变窄"(宽转长),unstack 让表"变矮变宽"(长转宽)。弄反了形状就不对,动手前先想清楚要哪种格式。

### ⚠ 04. 行业中性化前没对齐行业标签

如果某只票的行业标签缺失或拼错,groupby 会把它单独分一组或归错组,中性化结果就失真。先确认每只票都有正确的行业标签。

### ⚠ 05. pivot_table 默认聚合是均值

不指定 aggfunc 时默认取平均,想要求和、计数得显式写明,否则汇总口径就错了。

## 实战 SOP · 多重索引与分组实战 7 条 SOP

1. 面板数据优先用 MultiIndex — 多只票多个日期,用(日期, 股票)双层索引,分组和对齐都干净。
2. groupby 3 步法记牢 — split 按什么分、apply 每组算什么、combine 怎么拼回,写之前先想清这3 步。
3. 分组必跟聚合 — groupby 后立刻接 mean/sum/agg,别让分组对象悬在半空。
4. 长格式存储、宽格式展示 — 存盘和分组用长格式,画图和看走势用 unstack 成宽格式。
5. 算因子先行业中性化 — 剔除行业整体涨跌,留下个股相对同行的超额,排名才公允。
6. 透视先想清行列和口径 — pivot_table 动手前定好 index、columns、aggfunc 三件事。
7. 分组前先核对分组键 — 确认行业标签等分组键无缺失、无错拼,否则分组结果不可信。

> 把这段打印贴在你电脑边。

## 总结 · 你应该带走的

2. MultiIndex 多层索引天生适合"多只票 × 多个日期"的面板数据
3. groupby 是 split-apply-combine:按标签拆开、每堆各算、再拼回
4. stack 宽转长、unstack 长转宽,两种格式按需切换
5. pivot_table 一句话做出"行业 × 指标"的透视汇总
6. 行业中性化剔除赛道整体涨跌,衡量个股自己的真本事
7. 行业不同不等于分散,集中度和相关性才是分散的真标准
8. groupby 后必接聚合、中性化前先对齐行业标签

## 自测题

**Q1.** groupby 的 split-apply-combine 三步分别指什么?(提示:split 按某个标签把数据拆成几组(如按行业),apply 对每组各自做计算(如算平均收益),combine 把每组结果拼回一张表。)

**Q2.** stack 和 unstack 分别把表变成什么样?(提示:stack 把宽表(每列一只票)叠成长表,变高变窄;unstack 把长表摊成宽表,变矮变宽。长格式适合分组存储,宽格式适合画图。)

**Q3.** 为什么算选股因子前要做行业中性化?(提示:同行业的票会一起涨跌,这部分不算个股本事。减去所在行业平均后,剩下的超额才反映个股相对同行的真实强弱,排名才不会被强势行业整体带偏。)

**Q4.** 老周买了八只分属四个行业的票,为什么还说他没分散?(提示:因为八只里有五只挤在白酒和新能源两个赛道,行业集中度很高。分散看的是集中度和相关性,不是单纯的行业个数。)

把答案写下来,3 天后再回看。

## 下一节预告

**Day 038 · 缺失值处理** (Missing Data)

第38 课讲缺失值处理:NaN 到底是什么、怎么用 ffill 和 interpolate 填补行情缺口,以及一个能让回测虚假平稳的致命大坑——停牌期千 万不能简单前向填充。

## 推荐阅读

- Wes McKinney《Python for Data Analysis》(2022, O'Reilly)— pandas 作者亲笔,groupby 与 pivot_table 章节是最权威的入门
- pandas 官方文档《Group by: split-apply-combine》(2024)— 把分组3 步讲得最清楚,例子可直接照搬
- pandas 官方文档《Reshaping and pivot tables》(2024)— stack/unstack/pivot_table 的标准参考
- Grinold & Kahn《Active Portfolio Management》(1999, McGraw-Hill)— 行业中性化与因子构建的经典理论源头
- BARRA《风险模型手册》(2008)— 行业因子与中性化在真实风控里怎么用,工业界标准做法